In [ ]:
import os, copy
import glob

import numpy as np
import pandas as pd
import xarray as xr

import datetime as dt

import matplotlib
import matplotlib.pyplot as plt
#import colorcet as cc


opj = os.path.join


### set file path

In [ ]:
file = '/data/satellite/prisma/touria/L2A/bassas/PRS_L2A_hgrs_20240316074117_20240316074121_0001.nc'
file = '/data/satellite/prisma/touria/L2A/mayotte/PRS_L2A_hgrs_20240604071915_20240604071919_0001.nc'
#file = '/data/satellite/prisma/touria/L2A/glorieuse/PRS_L2A_hgrs_20240529071520_20240529071524_0001.nc'


### load image

In [ ]:
img =xr.open_dataset(file)
date = dt.datetime.strptime(img.acquisition_date,'%Y-%m-%dT%H:%M:%S.%f')
#img.acquisition_date
img.load()

### Plot spectral  Rrs

In [ ]:
img.Rrs.sel(wl=[440,510,565,610,670,705],method="nearest").plot(col='wl',col_wrap=3,size=5,aspect=1.2,robust=True,vmin=0,cmap=plt.cm.Spectral_r)

### Plot other parameters

In [ ]:
params=['aot_ref','aot_ref_std','tcwv','tcwv_std','brdfg','brdfg_std']

fig,axs = plt.subplots(2,3,figsize=(20,10))
axs=axs.ravel()

for i in range(len(params)):
    img[params[i]].plot.imshow(cmap=plt.cm.Spectral_r, robust=True,vmin=0,#vmax=0.201,
                               cbar_kwargs={'shrink': 0.78,'label':params[i]},ax=axs[i]) # extent=extent_val, transform=proj, 
    axs[i].set(xticks=[], yticks=[])
    axs[i].set_ylabel('')
    axs[i].set_xlabel('')    
    axs[i].set_title(params[i])    

In [ ]:
params=['aot_ref_full','tcwv_full','brdfg_full']

fig,axs = plt.subplots(1,3,figsize=(18,4))
axs=axs.ravel()

for i in range(len(params)):
    img[params[i]].plot.imshow(cmap=plt.cm.Spectral_r, robust=True,vmin=0,#vmax=0.201,
                               cbar_kwargs={'shrink': 0.78,'label':params[i]},ax=axs[i]) # extent=extent_val, transform=proj, 
    axs[i].set(xticks=[], yticks=[])
    axs[i].set_ylabel('')
    axs[i].set_xlabel('')    
    axs[i].set_title(params[i])    

In [ ]:
gamma=1#.5

fig,axs = plt.subplots(1,2,figsize=(20,10))
axs=axs.ravel()

rgb=img.Rrs.isel(wl=[30,20,6])
adj = xr.DataArray([1.2,1,1],coords={"wl":rgb.wl})
fig = ((rgb*adj)**gamma).plot.imshow(rgb='wl',robust=True,ax=axs[0])#, subplot_kws=dict(projection= l1c.proj))
img['brdfg_full'].plot.imshow(cmap=plt.cm.gray, robust=True,vmin=0,ax=axs[1],add_colorbar=False)

for i in range(2):
    axs[i].set(xticks=[], yticks=[])
    axs[i].set_ylabel('')
    axs[i].set_xlabel('')    
plt.tight_layout()        

## Plot and interact

In [ ]:
from holoviews import streams, opts
import holoviews as hv

import panel as pn
import param
# set the parameter for spectra extraction
hv.extension('bokeh')
pn.extension()

opts.defaults(
    opts.GridSpace(shared_xaxis=True, shared_yaxis=True),
    opts.Image(cmap='binary_r', width=800, height=700),
    opts.Labels(text_color='white', text_font_size='8pt', text_align='left', text_baseline='bottom'),
    opts.Path(color='white'),
    opts.Spread(width=900),
    opts.Overlay(show_legend=True))

In [ ]:
param = 'Rrs' #Rtoa'
#img = prod[['Rtoa','Ltoa']] 
raster = img[param]#L2grs #masked[param] 

#param = 'rho'
#raster = dc_l2c[param] 
cmap='Spectral_r'
#cmap='RdBu_r'
third_dim = 'wl'

wl= raster.wl.data
Nwl = len(wl)
ds = hv.Dataset(raster.persist())
im= ds.to(hv.Image, ['x', 'y'], dynamic=True).opts(cmap= cmap,colorbar=True,clim=(0,0.041)).hist(bin_range=(0,0.01)) 

polys = hv.Polygons([])
box_stream = hv.streams.BoxEdit(source=polys)
dmap, dmap_std=[],[]

def roi_curves(data,ds=ds):    
    if not data or not any(len(d) for d in data.values()):
        return hv.NdOverlay({0: hv.Curve([],'Wavelength (nm)', param)})

    curves,envelope = {},{}
    data = zip(data['x0'], data['x1'], data['y0'], data['y1'])
    for i, (x0, x1, y0, y1) in enumerate(data):
        selection = ds.select(x=(x0, x1), y=(y0, y1))
        mean = selection.aggregate(third_dim, np.mean).data
        std = selection.aggregate(third_dim, np.std).data
        wl = mean.wl

        curves[i]= hv.Curve((wl,mean[param]),'Wavelength (nm)', param) 

    return hv.NdOverlay(curves)


# a bit dirty to have two similar function, but holoviews does not like mixing Curve and Spread for the same stream
def roi_spreads(data,ds=ds):    
    if not data or not any(len(d) for d in data.values()):
        return hv.NdOverlay({0: hv.Curve([],'Wavelength (nm)', param)})

    curves,envelope = {},{}
    data = zip(data['x0'], data['x1'], data['y0'], data['y1'])
    for i, (x0, x1, y0, y1) in enumerate(data):
        selection = ds.select(x=(x0, x1), y=(y0, y1))
        mean = selection.aggregate(third_dim, np.mean).data
        std = selection.aggregate(third_dim, np.std).data
        wl = mean.wl

        curves[i]=  hv.Spread((wl,mean[param],std[param]))#.opts(fill_alpha=0.3)

    return hv.NdOverlay(curves)

mean=hv.DynamicMap(roi_curves,streams=[box_stream])
std =hv.DynamicMap(roi_spreads, streams=[box_stream])    
hlines = hv.HoloMap({wl[i]: hv.VLine(wl[i]) for i in range(Nwl)},third_dim )


hv.output(widget_location='top_left')

# visualize and play
graphs = ((mean* std *hlines).relabel(param))
layout = (im * polys +graphs    ).opts(
    opts.Curve(width=750,height=500, framewise=True,xlim=(400,1100)), 
    opts.Polygons(fill_alpha=0.2, color='green',line_color='black'), 
    opts.VLine(color='black')).cols(2)
layout 

In [ ]:

x0=mean.streams[0].data['x0']
x1=mean.streams[0].data['x1']
y0=mean.streams[0].data['y0']
y1=mean.streams[0].data['y1']

fig, axs = plt.subplots(1,1, figsize=(10, 6))#,sharey=True

for idx in range(len(x0)):
    raster_clipped = img.Rrs.sel(x=slice(x1[idx],x0[idx]),y=slice(y1[idx],y0[idx]))
    stacked = raster_clipped.sel(wl=slice(400,1100)).stack(gridcell=["y", "x"]).dropna('gridcell',thresh=0)
    group_coord ='wl'
    stat_coord='gridcell'
    stats = xr.Dataset({'median':stacked.groupby(group_coord).median(stat_coord)})
    stats['q25'] = stacked.groupby(group_coord).quantile(0.25,dim=stat_coord)
    stats['q75'] = stacked.groupby(group_coord).quantile(0.75,dim=stat_coord)
    stats['min'] = stacked.groupby(group_coord).min(stat_coord)
    stats['max'] = stacked.groupby(group_coord).max(stat_coord)
    stats['mean'] = stacked.groupby(group_coord).mean(stat_coord)
    stats['std'] = stacked.groupby(group_coord).std(stat_coord)
    stats['pix_num'] = stacked.count(stat_coord)


    axs.plot(stats.wl,stats['median'],marker='o',ms=3)#,c='k')
    #axs.plot(stats.wl,stats['mean'],c='red',ls='--')
    axs.fill_between(stats.wl, stats['q25'], stats['q75'],alpha=0.3,color='grey')

axs.axhline(y=0,color='k',lw=1)
axs.minorticks_on()
axs.set_ylabel(r'$R_{rs}\ (sr^{-1})$')
axs.set_xlabel(r'$Wavelength\ (nm)$')
plt.show()